# Audio QC Visualization (qc_results/*.json)

This notebook loads QC JSON results and visualizes key metrics.
Thresholds are heuristic guidelines; tune them for your dataset.


In [ ]:
!uv add --dev matplotlib pandas seaborn plotly

In [35]:
from pathlib import Path
import json
import numpy as np

try:
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    import plotly.express as px
    from IPython.display import display, Markdown
except ImportError as e:
    raise SystemExit("Missing deps. Run: pip install pandas matplotlib seaborn plotly") from e

In [37]:
# Read directly from CSV (skip qc_results JSON for now)
csv_path = Path("./data/csv/noise_report_16000_with_class.csv")
if not csv_path.exists():
    csv_path = Path("./data/csv/noise_report.csv")

if not csv_path.exists():
    df = pd.read_csv(csv_path)
    raise FileNotFoundError(f"CSV not found: {csv_path}")

# Normalize column names (trim spaces / BOM issues)
df.columns = [str(c).strip() for c in df.columns]

def _norm_col(c):
    return ''.join(ch for ch in str(c).lower() if ch.isalnum())

# Auto-rename columns that match result_mask by normalization
target = _norm_col('result_mask')
if 'result_mask' not in df.columns:
    for c in list(df.columns):
        if _norm_col(c) == target:
            df = df.rename(columns={c: 'result_mask'})
            break

# Normalize result_mask into a readable label (Success / Missing / Other)
if "result_mask" in df.columns:
    df["result_mask"] = df["result_mask"].fillna("Unknown").astype(str)

    def _normalize_mask(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "Unknown"
        if isinstance(v, bool):
            return "Success" if v else "Missing"
        if isinstance(v, (int, float)):
            if v == 1:
                return "Success"
            if v == 0:
                return "Missing"
            return str(v)
        s = str(v).strip().lower()
        if s in {"success", "pass", "ok", "true", "1"}:
            return "Success"
        if s in {"missing", "fail", "failed", "error", "0"}:
            return "Missing"
        return str(v)

    df["result_mask_label"] = df["result_mask"].apply(_normalize_mask)

# Convert numeric-like columns to numeric, keep metadata as strings
non_numeric = {"file_name", "file_path", "format", "subtype", "flags", "result_mask", "result_mask_label"}
for col in df.columns:
    if col not in non_numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df


,avg_speech_segment_s_any,avg_speech_segment_s_ch0,avg_speech_segment_s_ch1,channel_corr_01,channels,clipping_pct_ch0,clipping_pct_ch1,clipping_pct_max,crest_db_mean,crosstalk_db_01,...,spectral_rolloff95_hz_noise,spectral_rolloff95_hz_speech,speech_dropout_ratio_proxy,speech_ratio_any,speech_ratio_ch0,speech_ratio_ch1,subtype,zero_pct_max,result_mask_label,file_label
0,4.638008,6.620453,1.279180,0.000013,2,0,0,0,24.697083,0.000000,...,3341.079931,1543.409260,0.147670,0.777354,0.664490,0.129211,ALAW,0,Success Overmask,NaN
1,8.784841,9.892367,1.694836,-0.000038,2,0,0,0,21.929996,0.000000,...,3235.365452,1515.032288,0.112722,0.906478,0.825612,0.111862,ALAW,0,Success Overmask,NaN
2,15.410484,15.347144,2.009757,-0.000162,2,0,0,0,23.515561,0.000000,...,2371.428004,1655.336874,0.126926,0.938093,0.843820,0.320377,ALAW,0,Missing Mask,NaN
3,5.396024,6.370391,1.501174,0.000546,2,0,0,0,21.823191,-0.280289,...,2690.146306,1482.741857,0.173837,0.838057,0.696493,0.185499,ALAW,0,Missing Mask,NaN
4,10.233438,11.903779,2.529035,-0.000122,2,0,0,0,23.015088,-5.642715,...,2551.321413,1599.097495,0.171733,0.898289,0.735006,0.195946,ALAW,0,No Card,NaN
5,10.832271,11.188183,1.782970,-0.000248,2,0,0,0,26.182980,-11.628628,...,1506.342087,1750.337779,0.096519,0.918727,0.814387,0.213589,ALAW,0,Success Overmask,NaN
6,9.845186,12.361530,1.268677,-0.000072,2,0,0,0,25.840348,0.000000,...,2101.992805,1389.962599,0.110388,0.903312,0.823851,0.114842,ALAW,0,Success Overmask,NaN
7,6.212552,9.225863,1.163852,-0.000406,2,0,0,0,29.016632,-4.771217,...,2242.288451,1534.652135,0.114839,0.864951,0.771912,0.147688,ALAW,0,Success Overmask,NaN
8,9.738115,12.119886,1.234423,-0.000130,2,0,0,0,24.311516,0.000000,...,2477.472316,1479.934659,0.116119,0.915747,0.831942,0.145273,ALAW,0,Fail Overmask,NaN
9,6.708794,5.116024,2.383141,-0.000083,2,0,0,0,23.069513,0.000000,...,2168.354502,1621.068364,0.309276,0.851258,0.582855,0.397925,ALAW,0,Missing Mask,NaN


## Data coverage check (จำนวนข้อมูลที่ใช้จริงในกราฟ)
กราฟไม่ได้จำกัดจำนวนแถว แต่จะใช้เฉพาะค่าที่เป็นตัวเลขและไม่ใช่ NaN


In [38]:
display(Markdown(f'**Total rows:** {len(df)}'))
check_cols = ["speech_ratio_any", "est_snr_db_best", "clipping_pct_max", "longest_zero_run_ms_max", "lufs_i"]
for c in check_cols:
    if c in df.columns:
        non_null = df[c].notna().sum()
        unique = df[c].nunique(dropna=True)
        display(Markdown(f'- **{c}**: non‑null={non_null}, unique={unique}'))
    else:
        display(Markdown(f'- **{c}**: column not found'))


**Total rows:** 47

- **speech_ratio_any**: non‑null=47, unique=47

- **est_snr_db_best**: non‑null=47, unique=47

- **clipping_pct_max**: non‑null=47, unique=1

- **longest_zero_run_ms_max**: non‑null=47, unique=1

- **lufs_i**: non‑null=47, unique=47

## Result mask summary (Success / Missing / Other)
ดูจำนวนไฟล์แต่ละสถานะ และตารางไฟล์ที่ไม่ผ่านเพื่อดูสาเหตุ (flags/metrics)


In [39]:
if "result_mask" in df.columns:
    display(Markdown('**result_mask (raw) counts:**'))
    display(df["result_mask"].value_counts(dropna=False))
    if "result_mask_label" in df.columns:
        display(Markdown('**result_mask_label (normalized) counts:**'))
        display(df["result_mask_label"].value_counts(dropna=False))

    def _is_success(v):
        s = str(v).lower()
        return "success" in s

    cols = [c for c in ["file_name", "result_mask", "result_mask_label", "flags", "speech_ratio_any", "est_snr_db_best", "lufs_i", "clipping_pct_max", "longest_zero_run_ms_max"] if c in df.columns]
    df_fail = df[~df["result_mask"].apply(_is_success)][cols]
    display(df_fail)
else:
    display(Markdown('**result_mask ยังไม่มี** — ชื่อคอลัมน์ในไฟล์อาจมีช่องว่าง/ตัวพิมพ์ต่างกัน'))
    display(Markdown('**Available columns:** ' + ', '.join(df.columns)))


**result_mask (raw) counts:**

result_mask
Success Overmask    19
Missing Mask         9
No Card              8
Fail Overmask        6
Success Partial      4
Success Mask         1
Name: count, dtype: int64

**result_mask_label (normalized) counts:**

result_mask_label
Success Overmask    19
Missing Mask         9
No Card              8
Fail Overmask        6
Success Partial      4
Success Mask         1
Name: count, dtype: int64

,file_name,result_mask,result_mask_label,flags,speech_ratio_any,est_snr_db_best,lufs_i,clipping_pct_max,longest_zero_run_ms_max
2,219046816_18905173.wav,Missing Mask,Missing Mask,high_overlap_double_talk,0.938093,49.799927,-33.661652,0,0
3,232046522_18905229.wav,Missing Mask,Missing Mask,NaN,0.838057,50.011944,-34.685772,0,0
4,243103972_18462409.wav,No Card,No Card,NaN,0.898289,51.061829,-35.223919,0,0
8,243529025_18887462.wav,Fail Overmask,Fail Overmask,NaN,0.915747,50.927402,-35.119340,0,0
9,201487918_18847855.wav,Missing Mask,Missing Mask,NaN,0.851258,50.776230,-34.848900,0,0
15,232029987_18888694.wav,Missing Mask,Missing Mask,NaN,0.807126,43.607468,-37.770094,0,0
16,232030487_18889194.wav,No Card,No Card,NaN,0.849344,50.866058,-31.667580,0,0
17,232031656_18890363.wav,No Card,No Card,NaN,0.749168,50.609180,-36.887682,0,0
18,232047463_18906170.wav,Missing Mask,Missing Mask,NaN,0.824288,49.687988,-34.763421,0,0
19,130006857_18866317.wav,Missing Mask,Missing Mask,NaN,0.855119,49.341480,-34.815268,0,0


## File label and color mapping
Assign F1, F2, ... to each file and use consistent colors in plots.


In [40]:
def rgb_to_hex(c):
    return "#%02x%02x%02x" % (int(c[0] * 255), int(c[1] * 255), int(c[2] * 255))

if "file_name" in df.columns:
    df = df.copy()
    df["file_label"] = [f"F{i+1}" for i in range(len(df))]
    colors = sns.color_palette("tab10", n_colors=len(df))
    color_map = dict(zip(df["file_label"], colors))
    mapping_df = pd.DataFrame({
        "file_label": df["file_label"],
        "file_name": df["file_name"],
        "color": [rgb_to_hex(color_map[l]) for l in df["file_label"]],
    })
    mapping_df
else:
    color_map = None
    mapping_df = None
    None


## Guideline thresholds (heuristic)

- speech_ratio_any >= 0.15 (enough speech vs silence)
- est_snr_db_best >= 10 dB (SNR)
- clipping_pct_max <= 0.10 (% near full-scale)
- longest_zero_run_ms_max <= 500 ms (dropouts)
- lufs_i between -40 and -12 (too quiet / too loud)


In [41]:
GUIDELINES = {
    "speech_ratio_any": {"min": 0.15},
    "est_snr_db_best": {"min": 10.0},
    "clipping_pct_max": {"max": 0.10},
    "longest_zero_run_ms_max": {"max": 500.0},
    "lufs_i": {"min": -40.0, "max": -12.0},
}

def eval_range(series, min_val=None, max_val=None):
    ok = pd.Series(True, index=series.index)
    if min_val is not None:
        ok &= series >= min_val
    if max_val is not None:
        ok &= series <= max_val
    return np.where(series.isna(), "unknown", np.where(ok, "ok", "fail"))

summary_cols = []
if "file_name" in df:
    summary_cols.append("file_name")
if "file_label" in df:
    summary_cols.append("file_label")
summary = pd.DataFrame({c: df[c] for c in summary_cols})

for key, rule in GUIDELINES.items():
    if key in df:
        summary[key] = df[key]
        summary[f"{key}_ok"] = eval_range(df[key], rule.get("min"), rule.get("max"))

ok_cols = [c for c in summary.columns if c.endswith("_ok")]

def overall_state(row):
    if (row == "fail").any():
        return "fail"
    if (row == "unknown").any():
        return "unknown"
    return "ok"

if ok_cols:
    summary["guideline_overall"] = summary[ok_cols].apply(overall_state, axis=1)

summary


,file_name,file_label,speech_ratio_any,speech_ratio_any_ok,est_snr_db_best,est_snr_db_best_ok,clipping_pct_max,clipping_pct_max_ok,longest_zero_run_ms_max,longest_zero_run_ms_max_ok,lufs_i,lufs_i_ok,guideline_overall
0,130023309_18882769.wav,F1,0.777354,ok,49.564459,ok,0,ok,0,ok,-33.652363,ok,ok
1,219040943_18899300.wav,F2,0.906478,ok,50.768610,ok,0,ok,0,ok,-34.976019,ok,ok
2,219046816_18905173.wav,F3,0.938093,ok,49.799927,ok,0,ok,0,ok,-33.661652,ok,ok
3,232046522_18905229.wav,F4,0.838057,ok,50.011944,ok,0,ok,0,ok,-34.685772,ok,ok
4,243103972_18462409.wav,F5,0.898289,ok,51.061829,ok,0,ok,0,ok,-35.223919,ok,ok
5,243503609_18862046.wav,F6,0.918727,ok,40.135803,ok,0,ok,0,ok,-33.441889,ok,ok
6,243526393_18884830.wav,F7,0.903312,ok,51.040951,ok,0,ok,0,ok,-34.989881,ok,ok
7,243528453_18886890.wav,F8,0.864951,ok,45.351511,ok,0,ok,0,ok,-40.631281,fail,fail
8,243529025_18887462.wav,F9,0.915747,ok,50.927402,ok,0,ok,0,ok,-35.119340,ok,ok
9,201487918_18847855.wav,F10,0.851258,ok,50.776230,ok,0,ok,0,ok,-34.848900,ok,ok


## Flags from QC
The script also produces a "flags" column. Use it as a quick triage list.


In [42]:
cols = [c for c in ["file_name", "file_label", "flags"] if c in df]
df[cols]


,file_name,file_label,flags
0,130023309_18882769.wav,F1,NaN
1,219040943_18899300.wav,F2,NaN
2,219046816_18905173.wav,F3,high_overlap_double_talk
3,232046522_18905229.wav,F4,NaN
4,243103972_18462409.wav,F5,NaN
5,243503609_18862046.wav,F6,NaN
6,243526393_18884830.wav,F7,NaN
7,243528453_18886890.wav,F8,too_quiet_lufs
8,243529025_18887462.wav,F9,NaN
9,201487918_18847855.wav,F10,NaN


## Distributions


### How to read these histograms
- X-axis is the metric value; Y-axis is count. Each color (F1/F2/...) is a file.
- The dashed lines are guideline thresholds. Points of mass left/right of a line indicate potential issues.
- Look for outliers (a bar far away from the main cluster) and skewed distributions.
- With only a few files, patterns are weak; use this mainly to spot extreme values.
- If KDE is disabled, it means too few/constant values; rely on bars only.


In [43]:
def _status_color_map(series):
    base = {
        'success': '#2ca02c',
        'missing': '#d62728',
        'fail': '#ff7f0e',
        'no card': '#9467bd',
        'unknown': '#7f7f7f',
    }
    colors = {}
    palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set3
    idx = 0
    for label in sorted(series.dropna().unique()):
        s = str(label).strip().lower()
        if 'missing' in s:
            colors[label] = base['missing']
        elif 'fail' in s:
            colors[label] = base['fail']
        elif 'no card' in s or 'nocard' in s:
            colors[label] = base['no card']
        elif 'success' in s or 'pass' in s or 'ok' in s:
            colors[label] = base['success']
        elif s in ('unknown', 'nan', 'none'):
            colors[label] = base['unknown']
        else:
            colors[label] = palette[idx % len(palette)]
            idx += 1
    return colors

def _get_color_config():
    if 'result_mask' in df.columns:
        return 'result_mask', _status_color_map(df['result_mask'])
    if 'result_mask_label' in df.columns:
        return 'result_mask_label', _status_color_map(df['result_mask_label'])
    if 'file_label' in df.columns and 'mapping_df' in globals() and mapping_df is not None:
        return 'file_label', dict(zip(mapping_df['file_label'], mapping_df['color']))
    return None, None

def hist_with_guides(col, title, min_val=None, max_val=None):
    if col not in df:
        return
    col_data = df[col].dropna()
    if col_data.empty:
        if 'display' in globals():
            display(Markdown(f'**{col}**: ไม่มีข้อมูลตัวเลข (NaN ทั้งหมด)'))
        return
    color_col, color_map = _get_color_config()
    if color_col:
        fig = px.histogram(
            df,
            x=col,
            color=color_col,
            color_discrete_map=color_map,
            opacity=0.6,
            nbins=20,
            barmode='overlay',
            title=title,
        )
    else:
        fig = px.histogram(df, x=col, nbins=20, title=title)
    if min_val is not None:
        fig.add_vline(x=min_val, line_dash='dash', line_color='red')
    if max_val is not None:
        fig.add_vline(x=max_val, line_dash='dash', line_color='orange')
    fig.update_layout(bargap=0.05)
    fig.show()

HIST_EXPLAIN_TH = {
    "speech_ratio_any": "อ่านค่า speech_ratio_any: ยิ่งใกล้ 1 ยิ่งมีช่วงพูดเยอะ ถ้ากองค่าต่ำกว่าเส้น 0.15 แปลว่าไฟล์เงียบ/พูดน้อย อาจไม่เหมาะกับ ASR",
    "est_snr_db_best": "อ่านค่า SNR: ยิ่งสูงเสียงพูดชัดกว่า noise ถ้าส่วนใหญ่ต่ำกว่า 10 dB ให้พิจารณา denoise หรือคัดไฟล์",
    "clipping_pct_max": "อ่านค่า clipping: ค่ายิ่งสูงยิ่งเสียงแตก ถ้าเกิน 0.10% บ่อย ๆ คือมี clipping ชัด",
    "longest_zero_run_ms_max": "อ่านค่า dropout: ช่วงเงียบยาวผิดปกติ ถ้าเกิน 500 ms บ่อย ๆ อาจมีเสียงขาดหรือ dead air",
    "lufs_i": "อ่านค่า LUFS: ช่วงเหมาะสมอยู่ระหว่าง -40 ถึง -12 ถ้าต่ำกว่า = เบาเกินไป สูงกว่า = ดังเกินไป",
}

plot_specs = [
    ("speech_ratio_any", 0.15, None),
    ("est_snr_db_best", 10.0, None),
    ("clipping_pct_max", None, 0.10),
    ("longest_zero_run_ms_max", None, 500.0),
    ("lufs_i", -40.0, -12.0),
]

for key, vmin, vmax in plot_specs:
    hist_with_guides(key, key, vmin, vmax)
    if 'display' in globals():
        text = HIST_EXPLAIN_TH.get(key, '')
        if text:
            display(Markdown(f'**คำอธิบาย (ภาษาไทย):** {text}'))


**คำอธิบาย (ภาษาไทย):** อ่านค่า speech_ratio_any: ยิ่งใกล้ 1 ยิ่งมีช่วงพูดเยอะ ถ้ากองค่าต่ำกว่าเส้น 0.15 แปลว่าไฟล์เงียบ/พูดน้อย อาจไม่เหมาะกับ ASR

**คำอธิบาย (ภาษาไทย):** อ่านค่า SNR: ยิ่งสูงเสียงพูดชัดกว่า noise ถ้าส่วนใหญ่ต่ำกว่า 10 dB ให้พิจารณา denoise หรือคัดไฟล์

**คำอธิบาย (ภาษาไทย):** อ่านค่า clipping: ค่ายิ่งสูงยิ่งเสียงแตก ถ้าเกิน 0.10% บ่อย ๆ คือมี clipping ชัด

**คำอธิบาย (ภาษาไทย):** อ่านค่า dropout: ช่วงเงียบยาวผิดปกติ ถ้าเกิน 500 ms บ่อย ๆ อาจมีเสียงขาดหรือ dead air

**คำอธิบาย (ภาษาไทย):** อ่านค่า LUFS: ช่วงเหมาะสมอยู่ระหว่าง -40 ถึง -12 ถ้าต่ำกว่า = เบาเกินไป สูงกว่า = ดังเกินไป

## Scatter plots


### How to read these scatter plots
- Each dot is one file; axes are two metrics. Color matches the file label (F1/F2/...)
- Dots far from the main cluster are outliers to review.
- If dots form a slope, it suggests a relationship (e.g., louder files may show better SNR).
- Use guideline lines to see if points fall into acceptable regions.
- Jitter is added in some plots to separate overlapping points.


In [44]:
def _status_color_map(series):
    base = {
        'success': '#2ca02c',
        'missing': '#d62728',
        'fail': '#ff7f0e',
        'no card': '#9467bd',
        'unknown': '#7f7f7f',
    }
    colors = {}
    palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set3
    idx = 0
    for label in sorted(series.dropna().unique()):
        s = str(label).strip().lower()
        if 'missing' in s:
            colors[label] = base['missing']
        elif 'fail' in s:
            colors[label] = base['fail']
        elif 'no card' in s or 'nocard' in s:
            colors[label] = base['no card']
        elif 'success' in s or 'pass' in s or 'ok' in s:
            colors[label] = base['success']
        elif s in ('unknown', 'nan', 'none'):
            colors[label] = base['unknown']
        else:
            colors[label] = palette[idx % len(palette)]
            idx += 1
    return colors

def _get_color_config():
    if 'result_mask' in df.columns:
        return 'result_mask', _status_color_map(df['result_mask'])
    if 'result_mask_label' in df.columns:
        return 'result_mask_label', _status_color_map(df['result_mask_label'])
    if 'file_label' in df.columns and 'mapping_df' in globals() and mapping_df is not None:
        return 'file_label', dict(zip(mapping_df['file_label'], mapping_df['color']))
    return None, None

SCATTER_EXPLAIN_TH = {
    "SNR vs LUFS": "อ่านกราฟ SNR vs LUFS: จุดที่อยู่ในช่วง LUFS [-40, -12] และ SNR >= 10 dB คือกลุ่มที่เสียงค่อนข้างดี (ดังพอดีและชัด) จุดนอกกรอบคือไฟล์ที่ควรตรวจ",
    "Duration vs Speech Ratio": "อ่านกราฟ Duration vs Speech Ratio: ไฟล์ยาวแต่ speech_ratio ต่ำแปลว่ามีช่วงเงียบเยอะ/hold/dead air สูง จุดที่ speech_ratio สูงและ duration พอเหมาะมักใช้ได้ดี",
}

def scatter_plot(x, y, title, x_min=None, x_max=None, y_min=None, y_max=None):
    color_col, color_map = _get_color_config()
    hover_cols = [c for c in ["file_name", "flags", "result_mask", "result_mask_label", "sample_rate", "channels", "duration_sec"] if c in df.columns]
    if color_col:
        fig = px.scatter(
            df,
            x=x,
            y=y,
            color=color_col,
            color_discrete_map=color_map,
            hover_data=hover_cols,
            title=title,
        )
    else:
        fig = px.scatter(df, x=x, y=y, hover_data=hover_cols, title=title)
    if x_min is not None:
        fig.add_vline(x=x_min, line_dash='dash', line_color='red')
    if x_max is not None:
        fig.add_vline(x=x_max, line_dash='dash', line_color='orange')
    if y_min is not None:
        fig.add_hline(y=y_min, line_dash='dash', line_color='red')
    if y_max is not None:
        fig.add_hline(y=y_max, line_dash='dash', line_color='orange')
    fig.show()

if {"lufs_i", "est_snr_db_best"}.issubset(df.columns):
    scatter_plot(
        x="lufs_i",
        y="est_snr_db_best",
        title="SNR vs LUFS",
        x_min=-40.0,
        x_max=-12.0,
        y_min=10.0,
    )
    if 'display' in globals():
        display(Markdown(f"**คำอธิบาย (ภาษาไทย):** {SCATTER_EXPLAIN_TH.get('SNR vs LUFS', '')}"))

if {"speech_ratio_any", "duration_sec"}.issubset(df.columns):
    scatter_plot(
        x="speech_ratio_any",
        y="duration_sec",
        title="Duration vs Speech Ratio",
        x_min=0.15,
    )
    if 'display' in globals():
        display(Markdown(f"**คำอธิบาย (ภาษาไทย):** {SCATTER_EXPLAIN_TH.get('Duration vs Speech Ratio', '')}"))


**คำอธิบาย (ภาษาไทย):** อ่านกราฟ SNR vs LUFS: จุดที่อยู่ในช่วง LUFS [-40, -12] และ SNR >= 10 dB คือกลุ่มที่เสียงค่อนข้างดี (ดังพอดีและชัด) จุดนอกกรอบคือไฟล์ที่ควรตรวจ

**คำอธิบาย (ภาษาไทย):** อ่านกราฟ Duration vs Speech Ratio: ไฟล์ยาวแต่ speech_ratio ต่ำแปลว่ามีช่วงเงียบเยอะ/hold/dead air สูง จุดที่ speech_ratio สูงและ duration พอเหมาะมักใช้ได้ดี

## Interpretation notes
- Values outside guidelines are not automatically bad; treat them as candidates to review.
- Tune thresholds by inspecting distributions on 100-500 files (percentile-based cutoffs work well).
- If LUFS is NaN, scipy might be missing or the audio is too short.
